# Tool calling, raw

No LangChain. Just the `openai` SDK pointed at OpenRouter, to see the exact JSON going out and coming back.

In [ ]:
import json, os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(base_url=os.environ["OPENROUTER_BASE_URL"], api_key=os.environ["OPENROUTER_API_KEY"])
MODEL = os.environ["MODEL_NAME"]  # key is read by the client, never printed

## 1. The request

A tool is a JSON Schema describing a function. Same shape as your `SQLResponse`, wrapped in `type/function/parameters`. Building it costs nothing, so print it first.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_schema",
            "description": "List every table and its columns in the database.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_sample_values",
            "description": "Return up to 5 distinct values of one column, to see how data is written.",
            "parameters": {
                "type": "object",
                "properties": {
                    "table": {"type": "string"},
                    "column": {"type": "string"},
                },
                "required": ["table", "column"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "run_query",
            "description": "Run one read-only SELECT and return the rows.",
            "parameters": {
                "type": "object",
                "properties": {
                    "sql": {"type": "string"},
                },
                "required": ["sql"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "submit_sql",
            "description": "Submit the final SQL answer.",
            "parameters": {
                "type": "object",
                "properties": {
                    "sql": {"type": "string"},
                    "tables_used": {
                        "type": "array",
                        "items": {"type": "string"},
                    },
                    "reasoning": {"type": "string"},
                },
                "required": ["sql", "tables_used", "reasoning"],
            },
        },
    },
]

request = dict(
    model=MODEL,
    messages=[
        {"role": "user", "content": "How many customers are from Germany?"},
    ],
    tools=tools,
    tool_choice="auto",  # model decides; with 4 tools, forcing one would hide the choice
    temperature=0,
    max_tokens=300,
)
print(json.dumps(request, indent=2))

## 2. The call (PAID, a fraction of a cent)

Run only after approval.

In [ ]:
resp = client.chat.completions.create(**request)
print(resp.model_dump_json(indent=2, exclude_none=True))

## 3. Pull the arguments out

`arguments` is a JSON *string*, not a dict. The model wrote text; you parse it. Nothing has been executed.

In [ ]:
call = resp.choices[0].message.tool_calls[0]  # may be None if the model answered in prose
print(call.function.name, type(call.function.arguments))
args = json.loads(call.function.arguments)
print(args)